# ir_calendar_resend_api — 把已進 Databricks、沒進系統的批次補送

- 用途：`ir_calendar_consume_batches` 在 `api_base_url` 留空跑過的批次，資料已經在 Volume / bronze / silver，但沒 POST 到系統。這支把原本要送的 body 照**同一把 `Idempotency-Key`** 補送出去，效果與當初 consume 直接送完全等價。
- 什麼時候用這支、什麼時候倒游標重跑 consume：
  - **來源 VM 上批次還在** → 建議倒 `_cursor.json` 的 `last_seq` 重跑 consume，那是設計好的路徑
  - **來源已經清掉 / 不想重新下載整批** → 用這支，只打 API，不碰 Volume 與表
- 輸入：`b_{domain}_batch_log`（決定送哪幾批）、Volume `<volume_root>/ir_calendar/batches/<批次>/api/*.json` 或 `b_{domain}_record.payload`（body）
- 輸出：POST `<api_base_url>/companies/sync`、`<api_base_url>/ir-conferences/sync`；並把回應補寫回 `b_{domain}_batch_log.api_results`
- **不動**：Volume、bronze、silver、`_cursor.json` 一律不寫
- 參數（widgets）：見 [c01] 與下表
- 排程：不排程，人工執行
- 負責人 / 更新日期：（填）/ 2026-09-21

## 參數

| widget | 預設 | 說明 |
|---|---|---|
| `catalog` / `schema` / `domain` | `micenter` / `mi3_datahub_prod` / `ir_calendar` | 對照 `config/project.yml` |
| `volume_root` | `/Volumes/.../unstructured_data_file` | 歸檔 JSON 的位置 |
| `api_base_url` | 空（要送就必填） | 到 `/api/v1` 為止，不含結尾斜線 |
| `api_key_secret` | 空 | `scope/key`；沒填就不帶 `X-Api-Key`，系統多半會擋 |
| `verify_ssl` | `false` | 內網自簽憑證 |
| `batch_ids` | 空 | 逗號分隔指定批次；空 = 自動找 batch_log 裡 `SUCCESS` 且 `api_results` 為空的 |
| `payload_source` | `volume` | `volume`：Volume 歸檔；`bronze`：`b_*_record.payload`（Volume 被清掉時用） |
| `dry_run` | **`true`** | 預設不送。先跑一次看清單，確認沒問題再改 `false` |
| `force` | `false` | `true`：連已送過的也重送（同鍵，系統回上次結果） |
| `update_batch_log` | `true` | 把 `api_results` 補寫回 `batch_log` |
| `job_run_id` | 空 | job parameters 填 `{{job.run_id}}` |

## 一次執行做什麼

1. [c10] 從 `batch_log` 挑批次：`SUCCESS` 且（`api_results` 空 或 `force`），依 `seq` 由小到大
2. 每一批 [c11]：讀 manifest（Volume 優先，沒有就用 bronze 的 `batch_manifest` 列）→ 取 `record_type = api_payload` 的條目，**公司主檔排前面** → 逐個 POST，`Idempotency-Key = <批次>/<endpoint>`
3. [c06] 把 `api_results` MERGE 回 `batch_log`（只 update 這一欄，不新增列）
4. [c20] 任何一支回 `failed > 0` 或 `success = false` 就 raise
5. [c30] 把這次的結果攤平印出來

## 為什麼補送是安全的

`Idempotency-Key` 與當初 consume 會用的完全相同（`[c03] idem_key`），系統端同鍵重送直接回上次結果。所以：這支跑過之後就算再倒游標重跑 consume，系統也不會收到第二份。

## 安全設計

- `dry_run` 預設 `true`，Run All 不會意外把資料送出去
- 已送過的批次預設跳過，要 `force = true` 才重送
- API key 只從 `dbutils.secrets` 取，不出現在 code 或 widget

Cell 標籤規則見 `docs/conventions.md`。


In [ ]:
# [c01] params
# 補送：把已經在 Databricks、但沒送進系統的批次，照原本的 body 與 Idempotency-Key POST 出去。
# 不動 Volume、不動 bronze / silver、不動游標。唯一會寫的是 batch_log 的 api_results 欄（update_batch_log = true 時）。
# dry_run 預設 true：Run All 不會真的送出去，先看清單再改成 false。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")
dbutils.widgets.text("volume_root", "/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file")
dbutils.widgets.text("api_base_url", "")           # 例 https://<host>/api/v1，不含結尾斜線
dbutils.widgets.text("api_key_secret", "")         # "scope/key"，憑證只走 dbutils.secrets
dbutils.widgets.text("verify_ssl", "false")
dbutils.widgets.text("batch_ids", "")              # 逗號分隔；空 = 自動找 batch_log 裡 SUCCESS 但沒送過的
dbutils.widgets.text("payload_source", "volume")   # volume：Volume 歸檔的 api/*.json；bronze：b_*_record.payload
dbutils.widgets.text("dry_run", "true")            # true：只印會送什麼，不發 request
dbutils.widgets.text("force", "false")             # true：連已送過的批次也重送（同鍵，系統回上次結果）
dbutils.widgets.text("update_batch_log", "true")   # 把 api_results 補寫回 batch_log
dbutils.widgets.text("job_run_id", "")


def _flag(name: str) -> bool:
    return dbutils.widgets.get(name).strip().lower() == "true"


settings = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "volume_root": dbutils.widgets.get("volume_root").strip().rstrip("/"),
    "api_base_url": dbutils.widgets.get("api_base_url").strip().rstrip("/"),
    "api_key_secret": dbutils.widgets.get("api_key_secret").strip(),
    "verify_ssl": _flag("verify_ssl"),
    "batch_ids": [b.strip() for b in dbutils.widgets.get("batch_ids").split(",") if b.strip()],
    "payload_source": dbutils.widgets.get("payload_source").strip().lower(),
    "dry_run": _flag("dry_run"),
    "force": _flag("force"),
    "update_batch_log": _flag("update_batch_log"),
    "job_run_id": dbutils.widgets.get("job_run_id").strip() or None,
}
assert settings["catalog"] and settings["schema"] and settings["domain"], "catalog / schema / domain 不可為空"
assert settings["volume_root"].startswith("/Volumes/"), "volume_root 必須是 UC Volume 路徑"
assert settings["payload_source"] in ("volume", "bronze"), "payload_source 只能是 volume 或 bronze"
assert settings["dry_run"] or settings["api_base_url"], "要真的送就必須填 api_base_url（到 /api/v1 為止）"
print({k: v for k, v in settings.items() if k != "api_key_secret"})


In [ ]:
# [c02] imports
# 常數與 schema 必須與 ir_calendar_consume_batches 一致：ENDPOINT_BY_PATH 見那邊 [c02]，
# API_RESULT_SCHEMA 見 [c04]（欄位名與順序要與 b_*_batch_log.api_results 完全相同，否則 MERGE 會失敗）。
import json
import os
from datetime import datetime, timezone

import requests
import urllib3
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}
CONTROL_DIR = "ir_calendar"
ENDPOINT_BY_PATH = {
    "api/ir_conferences_sync.json": "ir-conferences/sync",
    "api/companies_sync.json": "companies/sync",
}
UTC = timezone.utc  # noqa: UP017 - 對齊 consume notebook 的寫法


def _s(name: str, dtype=None, nullable: bool = True) -> StructField:
    return StructField(name, dtype or StringType(), nullable)


API_RESULT_SCHEMA = StructType([
    _s("endpoint"), _s("idempotency_key"), _s("rows", IntegerType()), _s("http_status", IntegerType()),
    _s("success", BooleanType()), _s("created", IntegerType()), _s("updated", IntegerType()),
    _s("unchanged", IntegerType()), _s("failed", IntegerType()), _s("failures_json"), _s("response_json"),
    _s("posted_at", TimestampType()),
])
RESEND_SCHEMA = StructType([
    _s("batch_id", nullable=False), StructField("api_results", ArrayType(API_RESULT_SCHEMA)),
])


In [ ]:
# [c03] pure_helpers
# 純函式，無 I/O。api_result 與 ir_calendar_consume_batches [c05] 同一份，改那邊記得改這邊。


def table_name(settings: dict, layer: str, short: str) -> str:
    return f"{settings['catalog']}.{settings['schema']}.{LAYER_PREFIX[layer]}{settings['domain']}_{short}"


def to_int(v) -> int | None:
    return None if v is None or v == "" else int(v)


def js(obj) -> str | None:
    return None if obj is None else json.dumps(obj, ensure_ascii=False)


def endpoint_for(f: dict) -> str | None:
    """manifest 條目 → endpoint。優先用 manifest 帶的值，舊批次才落到路徑對照表。"""
    return f.get("endpoint") or ENDPOINT_BY_PATH.get(f["path"])


def send_order(f: dict) -> int:
    """公司主檔要先於場次：場次用 stock_code 找公司，公司還沒進去的話整列會失敗。"""
    return 0 if f["path"].startswith("api/companies") else 1


def idem_key(batch_id: str, endpoint: str) -> str:
    """與 consume [c10] 完全相同的組法：同一批同一支 endpoint 永遠同一把鍵。"""
    return f"{batch_id}/{endpoint}"


def api_result(*, endpoint: str, key: str, rows: int, http_status: int | None, body: dict | None,
               now: datetime) -> dict:
    data = (body or {}).get("data") or {}
    return {
        "endpoint": endpoint, "idempotency_key": key, "rows": rows, "http_status": http_status,
        "success": data.get("success"), "created": to_int(data.get("created")),
        "updated": to_int(data.get("updated")), "unchanged": to_int(data.get("unchanged")),
        "failed": to_int(data.get("failed")), "failures_json": js(data.get("failures")),
        "response_json": (js(body) or "")[:4000] or None, "posted_at": now,
    }


In [ ]:
# [c04] io_payload
# 取 manifest 與要送的 body。manifest 決定 endpoint，兩個來源都試：Volume 歸檔優先，拿不到就用 bronze 的
# batch_manifest 那列（bronze 是 append-only，Volume 被清掉時它還在）。body 則照 payload_source 決定從哪拿。


def archive_dir(settings: dict, batch_id: str) -> str:
    return f"{settings['volume_root']}/{CONTROL_DIR}/batches/{batch_id}"


def _bronze_payload(settings: dict, batch_id: str, batch_path: str) -> str | None:
    rows = (spark.table(table_name(settings, "bronze", "record"))
            .filter((F.col("batch_id") == F.lit(batch_id))
                    & (F.col("batch_path") == F.lit(batch_path)))
            .select("payload").limit(1).collect())
    return rows[0]["payload"] if rows else None


def load_manifest(settings: dict, batch_id: str) -> dict:
    path = f"{archive_dir(settings, batch_id)}/manifest.json"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    raw = _bronze_payload(settings, batch_id, "manifest.json")
    if raw:
        print(f"    manifest 取自 bronze（Volume 沒有 {path}）")
        return json.loads(raw)
    raise FileNotFoundError(f"{batch_id}：Volume 與 bronze 都找不到 manifest")


def load_payload(settings: dict, batch_id: str, rel_path: str) -> dict:
    """要 POST 的 body。volume：歸檔的 api/*.json；bronze：b_*_record.payload（同一份 JSON 全文）。"""
    if settings["payload_source"] == "volume":
        path = f"{archive_dir(settings, batch_id)}/{rel_path}"
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    raw = _bronze_payload(settings, batch_id, rel_path)
    if raw is None:
        raise FileNotFoundError(f"{batch_id}/{rel_path}：bronze 沒有這一列")
    return json.loads(raw)


def api_files(manifest: dict) -> list[dict]:
    """manifest 裡的 api_payload 條目，公司主檔排前面。"""
    files = [f for f in (manifest.get("files") or []) if f.get("record_type") == "api_payload"]
    return sorted(files, key=send_order)


In [ ]:
# [c05] io_api
# POST 出去。與 ir_calendar_consume_batches [c08] 同一份邏輯：同樣的 header、同樣的逾時、同樣的 raise_for_status。
if not settings["verify_ssl"]:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def api_key(settings: dict) -> str | None:
    ref = settings.get("api_key_secret") or ""
    if "/" not in ref:
        return None
    scope, key = ref.split("/", 1)
    return dbutils.secrets.get(scope=scope, key=key)


def api_post(settings: dict, endpoint: str, payload: dict, key: str) -> tuple[int | None, dict | None]:
    """回 (http_status, body)。dry_run 時不發 request，回 (None, None)。"""
    n_rows = len(payload.get("rows", []))
    url = f"{settings['api_base_url']}/{endpoint}"
    if settings["dry_run"]:
        print(f"    [dry-run] 會 POST {url}（{n_rows} 列，Idempotency-Key={key}）")
        return None, None
    headers = {"Content-Type": "application/json", "Idempotency-Key": key}
    secret = api_key(settings)
    if secret:
        headers["X-Api-Key"] = secret
    else:
        print("    警告：api_key_secret 沒設，這次不帶 X-Api-Key")
    r = requests.post(url, json=payload, headers=headers,
                      timeout=300, verify=settings["verify_ssl"])
    r.raise_for_status()
    body = r.json() if r.content else {}
    print(f"    POST {endpoint}: HTTP {r.status_code} {json.dumps(body, ensure_ascii=False)[:300]}")
    return r.status_code, body


In [ ]:
# [c06] io_batch_log
# 把 api_results 補寫回 b_*_batch_log。只 update 這一欄，不新增列：batch_log 沒有這批代表 consume 根本沒跑過，
# 那是另一個問題，這裡只印警告不掩蓋。
from delta.tables import DeltaTable


def update_api_results(settings: dict, batch_id: str, results: list[dict]) -> bool:
    table = table_name(settings, "bronze", "batch_log")
    if settings["dry_run"] or not settings["update_batch_log"]:
        print(f"    [skip] {table}: 不寫 api_results（dry_run 或 update_batch_log=false）")
        return False
    tgt = DeltaTable.forName(spark, table)
    row = [{"batch_id": batch_id, "api_results": results}]
    df = spark.createDataFrame(row, schema=RESEND_SCHEMA)
    (tgt.alias("t").merge(df.alias("s"), "t.batch_id = s.batch_id")
     .whenMatchedUpdate(set={"api_results": "s.api_results"})
     .execute())
    n = spark.table(table).filter(F.col("batch_id") == F.lit(batch_id)).count()
    if n == 0:
        print(f"    警告：{table} 沒有 {batch_id} 這一列，api_results 沒寫進去")
        return False
    print(f"    {table}: {batch_id} 的 api_results 已更新（{len(results)} 筆）")
    return True


In [ ]:
# [c10] pick_batches
# 決定要補送哪幾批。batch_ids 有填就照填的（仍需 status = SUCCESS）；沒填就自動找「SUCCESS 但 api_results 空」的。
# 已經送過的預設跳過，force = true 才重送（同一把 Idempotency-Key，系統會回上次結果，不會重複寫入）。
log_table = table_name(settings, "bronze", "batch_log")
log = spark.table(log_table).filter(F.col("status") == F.lit("SUCCESS"))
if settings["batch_ids"]:
    log = log.filter(F.col("batch_id").isin(settings["batch_ids"]))
rows = log.orderBy("seq").select("batch_id", "seq", "kind", "api_results").collect()   # 小表

if settings["batch_ids"]:
    missing = set(settings["batch_ids"]) - {r.batch_id for r in rows}
    assert not missing, f"這些 batch_id 在 batch_log 裡找不到或不是 SUCCESS：{sorted(missing)}"

todo, already = [], []
for r in rows:
    (already if (r.api_results and not settings["force"]) else todo).append(r)

print(f"[{log_table}] SUCCESS 批次 {len(rows)} 個")
for r in already:
    print(f"  跳過 seq={r.seq} {r.batch_id}（已送過 {len(r.api_results)} 筆，要重送請 force=true）")
for r in todo:
    print(f"  待送 seq={r.seq} {r.batch_id} kind={r.kind}")
assert todo, "沒有要送的批次：全部都送過了，或 batch_ids 篩掉了所有批次"


In [ ]:
# [c11] resend_batch
# 一批：讀 manifest → 取 api_payload 條目（公司主檔先）→ 逐個 POST → 回 api_results。
# 任一支失敗就丟例外，整批停住；已經送成功的那幾支不會被回滾，但重跑時同一把 Idempotency-Key 不會重複寫入。


def resend_batch(settings: dict, batch_id: str) -> list[dict]:
    manifest = load_manifest(settings, batch_id)
    files = api_files(manifest)
    print(f"  {batch_id}: kind={manifest.get('kind')} api_payload {len(files)} 個")
    if not files:
        print("    這批沒有 api_payload，沒東西可送")
        return []

    results = []
    for f in files:
        endpoint = endpoint_for(f)
        if not endpoint:
            raise ValueError(f"{batch_id}/{f['path']}：manifest 沒帶 endpoint，"
                             f"ENDPOINT_BY_PATH 也對不到，請補 [c02] 的對照")
        payload = load_payload(settings, batch_id, f["path"])
        key = idem_key(batch_id, endpoint)
        status, body = api_post(settings, endpoint, payload, key)
        if status is not None:
            results.append(api_result(endpoint=endpoint, key=key, rows=len(payload.get("rows", [])),
                                      http_status=status, body=body, now=datetime.now(UTC)))
    return results


In [ ]:
# [c20] main
# 依 seq 由小到大逐批送（批次之間也是先舊後新，公司主檔批次自然排在前面）。
# 有任何一批的 API 回 failed > 0 就在最後 raise，讓 job 顯示失敗；已送出的不會退回。
sent, problems = {}, []
for r in todo:
    results = resend_batch(settings, r.batch_id)
    if results:
        update_api_results(settings, r.batch_id, results)
    sent[r.batch_id] = results
    for a in results:
        if a["failed"] or a["success"] is False:
            problems.append(f"{r.batch_id} {a['endpoint']}: "
                            f"failed={a['failed']} {a['failures_json']}")

n_calls = sum(len(v) for v in sent.values())
print(f"\n完成：{len(sent)} 批，實際送出 {n_calls} 支"
      + ("（dry_run，沒有真的送）" if settings["dry_run"] else ""))
if problems:
    raise RuntimeError("系統 API 回報失敗：\n  - " + "\n  - ".join(problems))


In [ ]:
# [c30] check_results
# 收尾：把這次送的結果攤平出來看。dry_run 時沒東西可看。
if settings["dry_run"]:
    print("dry_run：沒有送出，也沒有寫 batch_log。確認上面的清單沒問題後把 dry_run 改成 false 再跑一次。")
elif sent:
    ids = list(sent)
    (spark.table(log_table).filter(F.col("batch_id").isin(ids))
     .select("batch_id", "seq", F.explode("api_results").alias("a"))
     .select("batch_id", "seq", "a.endpoint", "a.http_status", "a.success", "a.rows",
             "a.created", "a.updated", "a.unchanged", "a.failed", "a.posted_at")
     .orderBy("seq", "endpoint")
     .show(50, truncate=False))
    print("failed 有數字的話，看 api_results.failures_json；最常見是 stock_code 查無公司（系統不自動建公司）。")
